In [2]:
import numpy as np
import pandas as pd
import gensim
import networkx as nx
import pickle as pkl
from sklearn.neighbors import KDTree
from multiprocessing import Pool,cpu_count
import traceback
import time
from Bio import SeqIO
from karateclub.node_embedding.structural import Role2Vec

!python --version
print(f'gensim vesion {gensim.__version__}')

/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python 3.9.6
gensim vesion 4.4.0


In [3]:
mirna_model_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Modelos/mirna_precursor_doc2vec.model"
mirna_doc2vec_model = gensim.models.Doc2Vec.load(mirna_model_path)

mirna_especifico_model_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Modelos/mirna_especifico_doc2vec.model"
mirna_especifico_doc2vec_model = gensim.models.Doc2Vec.load(mirna_model_path)

lncrna_model_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Modelos/lncrna_doc2vec.model"
lncrna_doc2vec_model = gensim.models.Doc2Vec.load(lncrna_model_path)

In [4]:
def CTD(seq):
    n = float(len(seq))
    num_A, num_T, num_G, num_C = 0.0, 0.0, 0.0, 0.0
    AT_trans, AG_trans, AC_trans, TG_trans, TC_trans, GC_trans = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    for i in range(len(seq) - 1):
        if seq[i] == "A":
            num_A = num_A + 1
        if seq[i] == "T":
            num_T = num_T + 1
        if seq[i] == "G":
            num_G = num_G + 1
        if seq[i] == "C":
            num_C = num_C + 1
        if (seq[i] == "A" and seq[i + 1] == "T") or (seq[i] == "T" and seq[i + 1] == "A"):
            AT_trans = AT_trans + 1
        if (seq[i] == "A" and seq[i + 1] == "G") or (seq[i] == "G" and seq[i + 1] == "A"):
            AG_trans = AG_trans + 1
        if (seq[i] == "A" and seq[i + 1] == "C") or (seq[i] == "C" and seq[i + 1] == "A"):
            AC_trans = AC_trans + 1
        if (seq[i] == "T" and seq[i + 1] == "G") or (seq[i] == "G" and seq[i + 1] == "T"):
            TG_trans = TG_trans + 1
        if (seq[i] == "T" and seq[i + 1] == "C") or (seq[i] == "C" and seq[i + 1] == "T"):
            TC_trans = TC_trans + 1
        if (seq[i] == "G" and seq[i + 1] == "C") or (seq[i] == "C" and seq[i + 1] == "G"):
            GC_trans = GC_trans + 1

    a, t, g, c = 0, 0, 0, 0
    A0_dis, A1_dis, A2_dis, A3_dis, A4_dis = 0.0, 0.0, 0.0, 0.0, 0.0
    T0_dis, T1_dis, T2_dis, T3_dis, T4_dis = 0.0, 0.0, 0.0, 0.0, 0.0
    G0_dis, G1_dis, G2_dis, G3_dis, G4_dis = 0.0, 0.0, 0.0, 0.0, 0.0
    C0_dis, C1_dis, C2_dis, C3_dis, C4_dis = 0.0, 0.0, 0.0, 0.0, 0.0
    for i in range(len(seq) - 1):
        if seq[i] == "A":
            a = a + 1
            if a == 1:
                A0_dis = ((i * 1.0) + 1) / n
            if a == int(round(num_A / 4.0)):
                A1_dis = ((i * 1.0) + 1) / n
            if a == int(round(num_A / 2.0)):
                A2_dis = ((i * 1.0) + 1) / n
            if a == int(round((num_A * 3 / 4.0))):
                A3_dis = ((i * 1.0) + 1) / n
            if a == num_A:
                A4_dis = ((i * 1.0) + 1) / n
        if seq[i] == "T":
            t = t + 1
            if t == 1:
                T0_dis = ((i * 1.0) + 1) / n
            if t == int(round(num_T / 4.0)):
                T1_dis = ((i * 1.0) + 1) / n
            if t == int(round((num_T / 2.0))):
                T2_dis = ((i * 1.0) + 1) / n
            if t == int(round((num_T * 3 / 4.0))):
                T3_dis = ((i * 1.0) + 1) / n
            if t == num_T:
                T4_dis = ((i * 1.0) + 1) / n
        if seq[i] == "G":
            g = g + 1
            if g == 1:
                G0_dis = ((i * 1.0) + 1) / n
            if g == int(round(num_G / 4.0)):
                G1_dis = ((i * 1.0) + 1) / n
            if g == int(round(num_G / 2.0)):
                G2_dis = ((i * 1.0) + 1) / n
            if g == int(round(num_G * 3 / 4.0)):
                G3_dis = ((i * 1.0) + 1) / n
            if g == num_G:
                G4_dis = ((i * 1.0) + 1) / n
        if seq[i] == "C":
            c = c + 1
            if c == 1:
                C0_dis = ((i * 1.0) + 1) / n
            if c == int(round(num_C / 4.0)):
                C1_dis = ((i * 1.0) + 1) / n
            if c == int(round(num_C / 2.0)):
                C2_dis = ((i * 1.0) + 1) / n
            if c == int(round(num_C * 3 / 4.0)):
                C3_dis = ((i * 1.0) + 1) / n
            if c == num_C:
                C4_dis = ((i * 1.0) + 1) / n
    return (num_A / n, num_T / n, num_G / n, num_C / n,
            AT_trans / n - 1, AG_trans / (n - 1), AC_trans / (n - 1),
            TG_trans / n - 1, TC_trans / (n - 1), GC_trans / (n - 1),
            A0_dis, A1_dis, A2_dis, A3_dis, A4_dis,
            T0_dis, T1_dis, T2_dis, T3_dis, T4_dis,
            G0_dis, G1_dis, G2_dis, G3_dis, G4_dis,
            C0_dis, C1_dis, C2_dis, C3_dis, C4_dis)

def k_mer(seq):
    def get_1mer(seq):
        A_count = seq.count("A")
        T_count = seq.count("T")
        C_count = seq.count("C")
        G_count = seq.count("G")
        return [A_count/len(seq), T_count/len(seq), C_count/len(seq), G_count/len(seq)]

    def get_2mer(seq):
        res_dict = {}
        for x in "ATCG":
            for y in "ATCG":
                k = x + y
                res_dict[k] = 0
        i = 0
        while i + 2 < len(seq):
            k = seq[i:i + 2]
            i = i + 1
            res_dict[k] = res_dict[k] + 1
        return [x/len(seq) for x in list(res_dict.values())]

    def get_3mer(seq):
        res_dict = {}
        for x in "ATCG":
            for y in "ATCG":
                for z in "ATCG":
                    k = x + y + z
                    res_dict[k] = 0
        i = 0
        while i + 3 < len(seq):
            k = seq[i:i + 3]
            i = i + 1
            res_dict[k] = res_dict[k] + 1
        return [x/len(seq) for x in list(res_dict.values())]

    def get_4mer(seq):
        res_dict = {}
        for x in "ATCG":
            for y in "ATCG":
                for z in "ATCG":
                    for p in "ATCG":
                        k = x + y + z + p
                        res_dict[k] = 0
        i = 0
        while i + 4 < len(seq):
            k = seq[i:i + 4]
            i = i + 1
            res_dict[k] = res_dict[k] + 1
        return [x/len(seq) for x in list(res_dict.values())]
    return get_1mer(seq) + get_2mer(seq) + get_3mer(seq) + get_4mer(seq)

def segment(seq):
    res = []
    i = 0
    while i + 3 < len(seq):
        tmp = seq[i:i + 3]
        res.append(tmp)
        i = i + 1
    return res

def mirna_doc2vec_embedding(seq):
    seg = segment(seq)
    mirna_doc2vec_model.random.seed(0)
    vec = mirna_doc2vec_model.infer_vector(seg)
    return vec

def mirna_especifico_doc2vec_embedding(seq):
    seg = segment(seq)
    mirna_especifico_doc2vec_model.random.seed(0)
    vec = mirna_especifico_doc2vec_model.infer_vector(seg)
    return vec

def lncrna_doc2vec_embedding(seq):
    seg = segment(seq)
    lncrna_doc2vec_model.random.seed(0)
    vec = lncrna_doc2vec_model.infer_vector(seg)
    return vec

def read_fa(path):
    res={}
    rescords = list(SeqIO.parse(path,format="fasta"))
    for x in rescords:
        id = str(x.id)
        mayus=str(x.seq).upper()
        seq = str(mayus).replace("U","T").replace("N","")
        res[id]=seq
    return res

def to_dict(seq_dict,feature_list):
    res_dict = {}
    for i, k in enumerate(list(seq_dict.keys())):
        res_dict[k] = feature_list[i]
    return res_dict

def save_dict(x_dict,path):
    f = open(path,"w")
    for k,v in x_dict.items():
        tmp = k+","+",".join([str(x) for x in v])
        f.write(tmp+"\n")
    f.close()

def load_dict(path):
    lines = open(path,"r").readlines()
    res = {}
    for line in lines:
        x_list = line.strip().split(",")
        id = str(x_list[0])
        vec = [float(x) for x in x_list[1:]]
        res[id]=vec
    return res

In [5]:
lncrna_dict = read_fa("/Users/andrescubillovillalobos/Documents/CompGen_Inc/RedNeuronal1/data/fasta/fasta_seq/lncRNA.fa")
mirna_dict = read_fa("/Users/andrescubillovillalobos/Documents/CompGen_Inc/RedNeuronal1/data/fasta/fasta_seq/miRNA.fa")
mirna_especifico_dict = read_fa("/Users/andrescubillovillalobos/Documents/CompGen_Inc/RedNeuronal1/data/fasta/fasta_seq/miRNA_Especifico.fa")
pairs_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Necesarios/lnc_mir_list.txt"
pairs = [[x.strip().split(",")[0],x.strip().split(",")[1]] for x in open(pairs_path,"r").readlines()]

In [6]:
def process_lncrna():
    try:
        print("LncRNA")

        # Obtener las secuencias
        seq_list = list(lncrna_dict.values())
        seq_keys = list(lncrna_dict.keys())

        # Procesar las secuencias individualmente
        lncrna_ctds = [CTD(seq) for seq in seq_list]

        # Convertir los resultados en un diccionario
        lnc_ctd_dict = to_dict(dict(zip(seq_keys, seq_list)), lncrna_ctds)

        # Guardar el diccionario resultante en un archivo
        save_dict(lnc_ctd_dict, "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/lnc_ctd_dict.txt")
        print('Diccionario guardado correctamente.')

    except Exception as e:
        print("An error occurred during processing:")
        print(traceback.format_exc())

# Llamar a la función para procesar las secuencias
process_lncrna()

LncRNA
Diccionario guardado correctamente.


In [7]:
def process_doc2vec_lncrna():
    try:
        print("LncRNA Doc2Vec")

        # Obtener las secuencias
        seq_list = list(lncrna_dict.values())
        seq_keys = list(lncrna_dict.keys())
        # Medir el tiempo de inicio
        start_time = time.time()

        # Aplicar lncrna_doc2vec_embedding a cada secuencia de manera secuencial
        lncrna_doc2vecs = [lncrna_doc2vec_embedding(seq) for seq in seq_list]

        # Medir el tiempo de finalización
        end_time = time.time()

        # Convertir los resultados en un diccionario
        lnc_doc2vec_dict = to_dict(dict(zip(seq_keys, seq_list)), lncrna_doc2vecs)

        # Guardar el diccionario resultante en un archivo
        save_dict(lnc_doc2vec_dict, "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/lnc_doc2vec_dict.txt")
        print('Diccionario guardado correctamente.')

        # Imprimir el tiempo de ejecución
        print(f'Tiempo de ejecución (Doc2Vec): {end_time - start_time} segundos')

    except Exception as e:
        print("An error occurred during processing:")
        print(traceback.format_exc())

process_doc2vec_lncrna()

LncRNA Doc2Vec


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Diccionario guardado correctamente.
Tiempo de ejecución (Doc2Vec): 107.48282885551453 segundos


In [8]:
def process_kmer_lncrna():
    try:
        print("LncRNA k-mer")

        # Obtener las secuencias
        seq_list = list(lncrna_dict.values())
        seq_keys = list(lncrna_dict.keys())

        # Medir el tiempo de inicio
        start_time = time.time()

        # Aplicar k_mer a cada secuencia de manera secuencial
        lncrna_kmers = [k_mer(seq) for seq in seq_list]

        # Medir el tiempo de finalización
        end_time = time.time()

        # Convertir los resultados en un diccionario
        lnc_kmer_dict = to_dict(dict(zip(seq_keys, seq_list)), lncrna_kmers)

        # Guardar el diccionario resultante en un archivo
        save_dict(lnc_kmer_dict, "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/lnc_kmer_dict.txt")
        print('Diccionario guardado correctamente.')

        # Imprimir el tiempo de ejecución
        print(f'Tiempo de ejecución (k-mer): {end_time - start_time} segundos')

    except Exception as e:
        print("An error occurred during processing:")
        print(traceback.format_exc())

process_kmer_lncrna()

LncRNA k-mer
Diccionario guardado correctamente.
Tiempo de ejecución (k-mer): 2.112687110900879 segundos


In [9]:
print("lncrna graph")
lnc_ctd_dict = load_dict("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/lnc_ctd_dict.txt")
lnc_doc2vec_dict = load_dict("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/lnc_doc2vec_dict.txt")
lnc_kmer_dict = load_dict("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/lnc_kmer_dict.txt")

lnc_feature_dict = {}
for i,k in enumerate(list(lnc_ctd_dict.keys())):
    v1=lnc_ctd_dict[k]
    v2=lnc_doc2vec_dict[k]
    v3=lnc_kmer_dict[k]
    vec = v1+v2+v3
    lnc_feature_dict[k]=vec

lnc_list = []
lnc_vec =[]
for k,v in lnc_feature_dict.items():
    lnc_list.append(k)
    lnc_vec.append(v)

lnc_vec = np.array(lnc_vec)
kdt = KDTree(lnc_vec, leaf_size=30, metric='euclidean')
k_near = kdt.query(lnc_vec, k=10, return_distance=False)

g = nx.Graph()
for lnc in lnc_list:
    near = k_near[lnc_list.index(lnc)][1:]
    for x in near:
        g.add_edge(x,lnc_list.index(lnc))

model = Role2Vec(dimensions=256, workers=cpu_count(),epochs=16)
model.fit(g)
embedding = model.get_embedding()
lnc_role2vec_embedding={}
for node_index in range(len(g.nodes)):
    print(lnc_list[node_index])
    lnc_role2vec_embedding[lnc_list[node_index]]=embedding[node_index,:]
save_dict(lnc_role2vec_embedding,"/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/lnc_role2vec_dict.txt")


lncrna graph
NONHSAT000043.2
NONHSAT000044.2
NONHSAT000173.2
NONHSAT000179.2
NONHSAT000201.2
NONHSAT000202.2
NONHSAT000203.2
NONHSAT000204.2
NONHSAT000228.2
NONHSAT000284.2
NONHSAT000530.2
NONHSAT000531.2
NONHSAT000532.2
NONHSAT000533.2
NONHSAT000534.2
NONHSAT000535.2
NONHSAT000536.2
NONHSAT000537.2
NONHSAT000538.2
NONHSAT000539.2
NONHSAT000540.2
NONHSAT000542.2
NONHSAT000543.2
NONHSAT000544.2
NONHSAT000545.2
NONHSAT000546.2
NONHSAT000834.2
NONHSAT000920.2
NONHSAT000923.2
NONHSAT001062.2
NONHSAT001116.2
NONHSAT001136.2
NONHSAT001159.2
NONHSAT001162.2
NONHSAT001209.2
NONHSAT001210.2
NONHSAT001211.2
NONHSAT001212.2
NONHSAT001258.2
NONHSAT001380.2
NONHSAT001460.2
NONHSAT001461.2
NONHSAT001462.2
NONHSAT001463.2
NONHSAT001464.2
NONHSAT001465.2
NONHSAT001466.2
NONHSAT001467.2
NONHSAT001468.2
NONHSAT001498.2
NONHSAT001952.2
NONHSAT001953.2
NONHSAT001955.2
NONHSAT001967.2
NONHSAT001968.2
NONHSAT001969.2
NONHSAT001970.2
NONHSAT001971.2
NONHSAT001972.2
NONHSAT001973.2
NONHSAT001974.2
NONHSAT0019

In [10]:
def process_mirna():
    try:
        print("miRNA")

        # Obtener las secuencias
        seq_list = list(mirna_dict.values())
        seq_keys = list(mirna_dict.keys())

        # Procesar las secuencias individualmente
        mirna_ctds = [CTD(seq) for seq in seq_list]

        # Convertir los resultados en un diccionario
        mir_ctd_dict = to_dict(dict(zip(seq_keys, seq_list)), mirna_ctds)

        # Guardar el diccionario resultante en un archivo
        save_dict(mir_ctd_dict, "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_ctd_dict.txt")
        print('Diccionario guardado correctamente.')

    except Exception as e:
        print("An error occurred during processing:")
        print(traceback.format_exc())

# Llamar a la función para procesar las secuencias
process_mirna()

miRNA
Diccionario guardado correctamente.


In [11]:
def process_doc2vec_mirna():
    try:
        print("miRNA Doc2Vec")

        # Obtener las secuencias
        seq_list = list(mirna_dict.values())
        seq_keys = list(mirna_dict.keys())
        # Medir el tiempo de inicio
        start_time = time.time()

        # Aplicar mirna_doc2vec_embedding a cada secuencia de manera secuencial
        mirna_doc2vecs = [mirna_doc2vec_embedding(seq) for seq in seq_list]

        # Medir el tiempo de finalización
        end_time = time.time()

        # Convertir los resultados en un diccionario
        mir_doc2vec_dict = to_dict(dict(zip(seq_keys, seq_list)), mirna_doc2vecs)

        # Guardar el diccionario resultante en un archivo
        save_dict(mir_doc2vec_dict, "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_doc2vec_dict.txt")
        print('Diccionario guardado correctamente.')

        # Imprimir el tiempo de ejecución
        print(f'Tiempo de ejecución (Doc2Vec): {end_time - start_time} segundos')

    except Exception as e:
        print("An error occurred during processing:")
        print(traceback.format_exc())

process_doc2vec_mirna()

miRNA Doc2Vec
Diccionario guardado correctamente.
Tiempo de ejecución (Doc2Vec): 0.22949004173278809 segundos


In [12]:
def process_kmer_mirna():
    try:
        print("miRNA k-mer")

        # Obtener las secuencias
        seq_list = list(mirna_dict.values())
        seq_keys = list(mirna_dict.keys())

        # Medir el tiempo de inicio
        start_time = time.time()

        # Aplicar k_mer a cada secuencia de manera secuencial
        mirna_kmers = [k_mer(seq) for seq in seq_list]

        # Medir el tiempo de finalización
        end_time = time.time()

        # Convertir los resultados en un diccionario
        mir_kmer_dict = to_dict(dict(zip(seq_keys, seq_list)), mirna_kmers)

        # Guardar el diccionario resultante en un archivo
        save_dict(mir_kmer_dict, "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_kmer_dict.txt")
        print('Diccionario guardado correctamente.')

        # Imprimir el tiempo de ejecución
        print(f'Tiempo de ejecución (k-mer): {end_time - start_time} segundos')

    except Exception as e:
        print("An error occurred during processing:")
        print(traceback.format_exc())

process_kmer_mirna()

miRNA k-mer
Diccionario guardado correctamente.
Tiempo de ejecución (k-mer): 0.0193939208984375 segundos


In [13]:

print("mirna graph")
mir_ctd_dict = load_dict("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_ctd_dict.txt")
mir_doc2vec_dict = load_dict("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_doc2vec_dict.txt")
mir_kmer_dict = load_dict("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_kmer_dict.txt")

mir_feature_dict = {}
for i,k in enumerate(list(mir_ctd_dict.keys())):
    v1=mir_ctd_dict[k]
    v2=mir_doc2vec_dict[k]
    v3=mir_kmer_dict[k]
    vec = v1+v2+v3
    mir_feature_dict[k]=vec

mir_list = []
mir_vec =[]
for k,v in mir_feature_dict.items():
    mir_list.append(k)
    mir_vec.append(v)

mir_vec = np.array(mir_vec)
kdt = KDTree(mir_vec, leaf_size=30, metric='euclidean')
k_near = kdt.query(mir_vec, k=10, return_distance=False)

g = nx.Graph()
for mir in mir_list:
    near = k_near[mir_list.index(mir)][1:]
    for x in near:
        g.add_edge(x,mir_list.index(mir))

model = Role2Vec(dimensions=256, workers=cpu_count(),epochs=16)
model.fit(g)
embedding = model.get_embedding()

mir_role2vec_embedding={}
for node_index in range(len(g.nodes)):
    mir_role2vec_embedding[mir_list[node_index]]=embedding[node_index,:]
save_dict(mir_role2vec_embedding,"/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_role2vec_dict.txt")

mirna graph


In [14]:
def process_mirna():
    try:
        print("miRNA Especifico")

        # Obtener las secuencias
        seq_list = list(mirna_especifico_dict.values())
        seq_keys = list(mirna_especifico_dict.keys())

        # Procesar las secuencias individualmente
        mirna_ctds = [CTD(seq) for seq in seq_list]

        # Convertir los resultados en un diccionario
        mir_ctd_dict = to_dict(dict(zip(seq_keys, seq_list)), mirna_ctds)

        # Guardar el diccionario resultante en un archivo
        save_dict(mir_ctd_dict, "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_especifico_ctd_dict.txt")
        print('Diccionario guardado correctamente.')

    except Exception as e:
        print("An error occurred during processing:")
        print(traceback.format_exc())

# Llamar a la función para procesar las secuencias
process_mirna()

miRNA Especifico
Diccionario guardado correctamente.


In [15]:
def process_doc2vec_mirna():
    try:
        print("miRNA especifico Doc2Vec")

        # Obtener las secuencias
        seq_list = list(mirna_especifico_dict.values())
        seq_keys = list(mirna_especifico_dict.keys())
        # Medir el tiempo de inicio
        start_time = time.time()

        # Aplicar mirna_doc2vec_embedding a cada secuencia de manera secuencial
        mirna_doc2vecs = [mirna_especifico_doc2vec_embedding(seq) for seq in seq_list]

        # Medir el tiempo de finalización
        end_time = time.time()

        # Convertir los resultados en un diccionario
        mir_doc2vec_dict = to_dict(dict(zip(seq_keys, seq_list)), mirna_doc2vecs)

        # Guardar el diccionario resultante en un archivo
        save_dict(mir_doc2vec_dict, "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_especifico_doc2vec_dict.txt")
        print('Diccionario guardado correctamente.')

        # Imprimir el tiempo de ejecución
        print(f'Tiempo de ejecución (Doc2Vec): {end_time - start_time} segundos')

    except Exception as e:
        print("An error occurred during processing:")
        print(traceback.format_exc())

process_doc2vec_mirna()

miRNA especifico Doc2Vec
Diccionario guardado correctamente.
Tiempo de ejecución (Doc2Vec): 1.4328370094299316 segundos


In [16]:
def process_kmer_mirna():
    try:
        print("miRNA especifico k-mer")

        # Obtener las secuencias
        seq_list = list(mirna_especifico_dict.values())
        seq_keys = list(mirna_especifico_dict.keys())

        # Medir el tiempo de inicio
        start_time = time.time()

        # Aplicar k_mer a cada secuencia de manera secuencial
        mirna_kmers = [k_mer(seq) for seq in seq_list]

        # Medir el tiempo de finalización
        end_time = time.time()

        # Convertir los resultados en un diccionario
        mir_kmer_dict = to_dict(dict(zip(seq_keys, seq_list)), mirna_kmers)

        # Guardar el diccionario resultante en un archivo
        save_dict(mir_kmer_dict, "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_especifico_kmer_dict.txt")
        print('Diccionario guardado correctamente.')

        # Imprimir el tiempo de ejecución
        print(f'Tiempo de ejecución (k-mer): {end_time - start_time} segundos')

    except Exception as e:
        print("An error occurred during processing:")
        print(traceback.format_exc())

process_kmer_mirna()

miRNA especifico k-mer
Diccionario guardado correctamente.
Tiempo de ejecución (k-mer): 0.12900614738464355 segundos


In [17]:

print("mirna especifico graph")
mir_ctd_dict = load_dict("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_especifico_ctd_dict.txt")
mir_doc2vec_dict = load_dict("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_especifico_doc2vec_dict.txt")
mir_kmer_dict = load_dict("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_especifico_kmer_dict.txt")

mir_feature_dict = {}
for i,k in enumerate(list(mir_ctd_dict.keys())):
    v1=mir_ctd_dict[k]
    v2=mir_doc2vec_dict[k]
    v3=mir_kmer_dict[k]
    vec = v1+v2+v3
    mir_feature_dict[k]=vec

mir_list = []
mir_vec =[]
for k,v in mir_feature_dict.items():
    mir_list.append(k)
    mir_vec.append(v)

mir_vec = np.array(mir_vec)
kdt = KDTree(mir_vec, leaf_size=30, metric='euclidean')
k_near = kdt.query(mir_vec, k=10, return_distance=False)

g = nx.Graph()
for mir in mir_list:
    near = k_near[mir_list.index(mir)][1:]
    for x in near:
        g.add_edge(x,mir_list.index(mir))

model = Role2Vec(dimensions=256, workers=cpu_count(),epochs=16)
model.fit(g)
embedding = model.get_embedding()

mir_role2vec_embedding={}
for node_index in range(len(g.nodes)):
    mir_role2vec_embedding[mir_list[node_index]]=embedding[node_index,:]
save_dict(mir_role2vec_embedding,"/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_especifico_role2vec_dict.txt")

mirna especifico graph
